# Experiment 4: Geographic Information

## Research Question

Does geographic information, represented by `latitude` and `longitude`,
improve the out-of-sample prediction of median house value?

## Hypothesis

Adding `latitude` and `longitude` will improve out-of-sample prediction
because the EDA revealed strong spatial patterns in housing values that
are not adequately captured by their individual Pearson correlations
with the target.

## Experimental Principle

The presence of geographic features will be changed while keeping the
following constant:

- Training data
- Validation data
- Test data
- Preprocessing strategy
- Model
- Model hyperparameters
- Evaluation metrics

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("E:\House Price Prediction ML Project\Data\Raw\housing.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
C:\Users\HP\AppData\Local\Temp\ipykernel_5216\4013469833.py:1: SyntaxWarning: invalid escape sequence '\H'
  df = pd.read_csv("E:\House Price Prediction ML Project\Data\Raw\housing.csv")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
X = df.drop(columns=["median_house_value"])
y =df['median_house_value']

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor


#Split the raw data inot train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

In [5]:
X_train_pool, X_val, y_train_pool, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

#x_train_pool, y_train_pool -> 80% of original training data, used to train machine learning model.
#x_val, y_val -> 20% of original training data (test_size=0.20), held back to tune hyperparameters and evaluate performance before final testing.


### 1. Create Feature sets

In [8]:
geographical_features = ['longitude', 'latitude']

feature_sets = {
    'without_geo_features': [i for i in X_train_pool.columns if i not in geographical_features],
    'with_geo_features': X_train_pool.columns.tolist()

}


In [9]:
for name, features in feature_sets.items():
    print(name)
    print(features)
    print()

without_geo_features
['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity']

with_geo_features
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity']



### 2. Seprate Numerical and Categorical Features in without geo features and with geo features

In [10]:
#For With geo features

numerical_features_with_geo = [i for i in feature_sets['with_geo_features'] if X_train_pool[i].dtype in ['int64', 'float64']]

categorical_features_with_geo = [i for i in feature_sets['with_geo_features'] if X_train_pool[i].dtype == 'object']

In [11]:
#For without geo

numerical_features_without_geo = [i for i in feature_sets['without_geo_features'] if X_train_pool[i].dtype in ['int64', 'float64']]

categorical_features_without_geo = [i for i in feature_sets['without_geo_features'] if X_train_pool[i].dtype == 'object']

### 3. Create Preprocessing pipeline

In [12]:
#Create a preprocessing pipeline for numerical and categorical features using SimpleImputer, StandardScaler, and OneHotEncoder.
numeric_transform = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transform = Pipeline([
    ('imputer', SimpleImputer(strategy ='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [13]:
#Apply the preprocessing pipeline to the training data using ColumnTransformer,
#which allows for different preprocessing steps to be applied to different subsets of features.

preprocessing_with_geo = ColumnTransformer([
    ('num', numeric_transform, numerical_features_with_geo),
    ('cat', categorical_transform, categorical_features_with_geo)
])

preprocessing_without_geo = ColumnTransformer([
    ('num', numeric_transform, numerical_features_without_geo),
    ('cat', categorical_transform, categorical_features_without_geo)
])

In [14]:
#create two separate pipelines for training a linear regression model, 
#one with geographical features and one without. 
#Each pipeline includes the appropriate preprocessing steps followed by the linear regression model.

model_with_geo = Pipeline([
    ('preprocessor', preprocessing_with_geo),
    ('regressor', LinearRegression())
])

model_without_geo = Pipeline([
    ('preprocessor', preprocessing_without_geo),
    ('regressor', LinearRegression())
])

### 4. Evaluate Model


In [15]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, X_train_pool, y_train_pool, X_val, y_val):
    # Fit the model on the training data
    model.fit(X_train_pool, y_train_pool)

    # Make predictions on the validation data
    y_train_pred = model.predict(X_train_pool)
    y_val_pred = model.predict(X_val)

    # Calculate evaluation metrics for both training and validation data
    results = {
        "Train_RMSE": root_mean_squared_error(
            y_train,
            y_train_pred
        ),

        "Validation_RMSE": root_mean_squared_error(
            y_val,
            y_val_pred
        ),

        "Train_MAE": mean_absolute_error(
            y_train,
            y_train_pred
        ),

        "Validation_MAE": mean_absolute_error(
            y_val,
            y_val_pred
        ),

        "Train_R2": r2_score(
            y_train,
            y_train_pred
        ),

        "Validation_R2": r2_score(
            y_val,
            y_val_pred
        )
    }

    return results


In [22]:
X_train_pool[feature_sets["with_geo_features"]].shape
y_train_pool.shape
X_val[feature_sets["without_geo_features"]].shape

(3303, 7)

In [ ]:
results = []

# Without geographic features
def evaluate_model(model, X_train_pool, y_train_pool, X_val, y_val):
    # Fit the model on the training data
    model.fit(X_train_pool, y_train_pool)

    # Make predictions on the validation data
    y_train_pred = model.predict(X_train_pool)
    y_val_pred = model.predict(X_val)

    # Calculate evaluation metrics for both training and validation data
    results = {
        "Train_RMSE": root_mean_squared_error(
            y_train_pool,
            y_train_pred
        ),

        "Validation_RMSE": root_mean_squared_error(
            y_val,
            y_val_pred
        ),

        "Train_MAE": mean_absolute_error(
            y_train_pool,
            y_train_pred
        ),

        "Validation_MAE": mean_absolute_error(
            y_val,
            y_val_pred
        ),

        "Train_R2": r2_score(
            y_train_pool,
            y_train_pred
        ),

        "Validation_R2": r2_score(
            y_val,
            y_val_pred
        )
    }

    return results

result_without_geo["Feature_Set"] = "without_geo_features"

results.append(result_without_geo)


# With geographic features
result_with_geo = evaluate_model(
    model_with_geo,
    X_train_pool[feature_sets["with_geo_features"]],
    y_train_pool,
    X_val[feature_sets["with_geo_features"]],
    y_val
)

result_with_geo["Feature_Set"] = "with_geo_features"

results.append(result_with_geo)

ValueError: Found input variables with inconsistent numbers of samples: [16512, 13209]